In [1]:
from datasets import load_dataset
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
import random
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


/home/flavien/anaconda3/envs/nlp_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# Load the dataset
dataset = load_dataset("jniimi/tripadvisor-review-rating")
raw_data = pd.DataFrame(dataset['train'])

201295

In [3]:
class ReviewDataset(Dataset):
    def __init__(self, dataframe, vocab=None):
        self.data = dataframe

        if vocab is None:
            self.build_vocab()
        else:
            self.vocab = vocab

    def build_vocab(self):
        text = ' '.join(self.data['review'])
        counter = Counter(text)
        self.vocab = {ch: idx+1 for idx, (ch, _) in enumerate(counter.most_common())}
        self.vocab['<PAD>'] = 0

    def encode(self, text):
        return [self.vocab.get(ch, 0) for ch in text]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        title_encoded = self.encode(row['title'])
        review_encoded = self.encode(row['review'])
        features = torch.tensor([
            row['overall'], row['cleanliness'], row['value'],
            row['location'], row['rooms'], row['sleep_quality']
        ], dtype=torch.float32) / 5.0  # Normalisation sur [0,1]
        return {
            'features': features,
            'title': torch.tensor(title_encoded, dtype=torch.long),
            'review': torch.tensor(review_encoded, dtype=torch.long)
        }


In [4]:
class ReviewGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)  # Embedding layer to convert input indices to dense vectors

        self.title_encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)  # LSTM to encode the title sequence
        
        self.fc_features = nn.Linear(6, hidden_size)  # Fully connected layer to process numerical features (6 features)

        self.lstm = nn.LSTM(embed_size + hidden_size * 2, hidden_size, num_layers, batch_first=True)  # LSTM for sequence generation
        self.fc_out = nn.Linear(hidden_size, vocab_size)  # Fully connected layer to map hidden states to vocabulary size

    def forward(self, features, title, review_in):
        embedded_title = self.embed(title)
        _, (h_title, _) = self.title_encoder(embedded_title)

        features_embedded = torch.relu(self.fc_features(features)).unsqueeze(0)

        context = torch.cat((h_title, features_embedded), dim=2)

        # Repeat context for each timestep
        context = context.repeat(review_in.size(1), 1, 1).permute(1, 0, 2)

        embedded_review = self.embed(review_in)
        lstm_input = torch.cat((embedded_review, context), dim=2)

        output, _ = self.lstm(lstm_input)
        logits = self.fc_out(output)

        return logits

    def generate(self, features, title, max_length=300, temperature=1.0):
        device = next(self.parameters()).device
        self.eval()

        with torch.no_grad():
            embedded_title = self.embed(title.unsqueeze(0))
            _, (h_title, _) = self.title_encoder(embedded_title)

            features_embedded = torch.relu(self.fc_features(features.unsqueeze(0))).unsqueeze(0)

            context = torch.cat((h_title, features_embedded), dim=2)
            context = context.repeat(1, 1, 1)

            input_char = torch.tensor([[self.embed.padding_idx]], device=device)

            generated = []

            hidden = None
            for _ in range(max_length):
                embedded_input = self.embed(input_char)
                lstm_input = torch.cat((embedded_input, context), dim=2)

                output, hidden = self.lstm(lstm_input, hidden)
                logits = self.fc_out(output.squeeze(1))

                probs = torch.softmax(logits / temperature, dim=-1)
                input_char = torch.multinomial(probs, num_samples=1)

                char_idx = input_char.item()
                if char_idx == self.embed.padding_idx:
                    break
                generated.append(char_idx)

            return generated


In [5]:
def collate_fn(batch):
    batch_features = torch.stack([item['features'] for item in batch])
    batch_titles = nn.utils.rnn.pad_sequence([item['title'] for item in batch], batch_first=True, padding_value=0)
    batch_reviews = nn.utils.rnn.pad_sequence([item['review'] for item in batch], batch_first=True, padding_value=0)

    return batch_features, batch_titles, batch_reviews[:, :-1], batch_reviews[:, 1:]


In [6]:
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001, device='cuda'):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for features, titles, inputs, targets in train_loader:
            features, titles, inputs, targets = features.to(device), titles.to(device), inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(features, titles, inputs)

            loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, titles, inputs, targets in val_loader:
                features, titles, inputs, targets = features.to(device), titles.to(device), inputs.to(device), targets.to(device)
                outputs = model(features, titles, inputs)
                loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs} -- Train Loss: {avg_train_loss:.4f} -- Val Loss: {avg_val_loss:.4f}")


In [4]:
# Charger le DataFrame
df = raw_data.drop(columns=['stay_year', 'post_date', 'freq', 'lang'])

# Drop the rows with missing User Prompt
df = df.dropna()

# Drop the duplicates
df = df.drop_duplicates()

# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

print(len(df))

201295


In [8]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = ReviewDataset(train_df)
val_dataset = ReviewDataset(val_df, vocab=train_dataset.vocab)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Initialiser modèle
vocab_size = len(train_dataset.vocab)
model = ReviewGenerator(vocab_size, embed_size=128, hidden_size=256)
print(model)

ReviewGenerator(
  (embed): Embedding(456, 128)
  (title_encoder): LSTM(128, 256, batch_first=True)
  (fc_features): Linear(in_features=6, out_features=256, bias=True)
  (lstm): LSTM(640, 256, batch_first=True)
  (fc_out): Linear(in_features=256, out_features=456, bias=True)
)


In [9]:
# Entraîner
train_model(model, train_loader, val_loader, epochs=20, lr=0.001, device=device)

KeyboardInterrupt: 

In [ ]:
# Exemples de données de génération
example_features = torch.tensor([5.0, 5.0, 5.0, 5.0, 5.0, 5.0]) / 5.0
example_title = torch.tensor(train_dataset.encode("Perfect stay"), dtype=torch.long)

generated_ids = model.generate(example_features.cuda(), example_title.cuda(), max_length=400)
id_to_char = {idx: ch for ch, idx in train_dataset.vocab.items()}
generated_text = ''.join([id_to_char.get(idx, '') for idx in generated_ids])

print(generated_text)